In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install torchviz

In [ ]:
import os
import glob
import json
import copy
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("[WARN] xgboost not installed. XGBoost baseline will be skipped.")

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "/content/drive/MyDrive/processed_meg_regression"

BATCH_SIZE = 8
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
RUN_SEED = 42

WINDOW_SIZE = 2048
WINDOW_STRIDE = 1024

EARLY_STOPPING_PATIENCE = 12
MIN_DELTA = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RUN_SEED)
np.random.seed(RUN_SEED)
random.seed(RUN_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# JSON HELPER
# =========================================================
def to_python(obj):
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_python(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_python(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

# =========================================================
# WINDOWING
# =========================================================
def extract_windows_from_trial(trial_x, trial_y, window_size=2048, stride=1024):
    c, t = trial_x.shape
    xs = []
    start = 0

    while start + window_size <= t:
        xs.append(trial_x[:, start:start + window_size])
        start += stride

    if len(xs) == 0:
        return None, None

    return np.stack(xs).astype(np.float32), np.array(trial_y, dtype=np.float32)

# =========================================================
# LOADING
# =========================================================
def extract_subject_id_from_path(x_path):
    base = os.path.basename(x_path)
    return base.split("_")[0]

def load_all_trials_with_subjects(data_dir):
    x_paths = sorted(glob.glob(os.path.join(data_dir, "*_X.npy")))
    print(f"Searching in: {data_dir}")
    print(f"Found {len(x_paths)} X files")

    seq_x_list, y_list, subject_list = [], [], []

    for x_path in x_paths:
        y_path = x_path.replace("_X.npy", "_y.npy")
        if not os.path.exists(y_path):
            print(f"[WARN] Missing y file for: {x_path}")
            continue

        subject_id = extract_subject_id_from_path(x_path)
        X = np.load(x_path)
        y = np.load(y_path)

        n = min(len(X), len(y))
        X, y = X[:n], y[:n]

        for i in range(n):
            seq_x, y_trial = extract_windows_from_trial(
                X[i], y[i],
                window_size=WINDOW_SIZE,
                stride=WINDOW_STRIDE
            )
            if seq_x is not None:
                seq_x_list.append(seq_x)
                y_list.append(y_trial)
                subject_list.append(subject_id)

    print(f"Total usable trial sequences: {len(seq_x_list)}")
    print(f"Total unique subjects: {len(set(subject_list))}")
    return seq_x_list, y_list, subject_list

def filter_by_subjects(seq_x_list, y_list, subject_list, allowed_subjects):
    out_x, out_y = [], []
    for x, y, s in zip(seq_x_list, y_list, subject_list):
        if s in allowed_subjects:
            out_x.append(x)
            out_y.append(y)
    return out_x, out_y

# =========================================================
# LABELS: AROUSAL ONLY
# =========================================================
def compute_train_arousal_threshold(train_y_list):
    Y_train = np.stack(train_y_list, axis=0)
    arousal_thresh = np.median(Y_train[:, 1])
    return float(arousal_thresh)

def make_arousal_binary_labels(y_list, arousal_thresh):
    Y = np.stack(y_list, axis=0)
    arousal_labels = (Y[:, 1] >= arousal_thresh).astype(np.int64)
    return arousal_labels

# =========================================================
# DATASET
# =========================================================
class TrialClassificationDataset(Dataset):
    def __init__(self, seq_x_list, labels):
        self.seq_x_list = seq_x_list
        self.labels = labels.astype(np.float32)

    def __len__(self):
        return len(self.seq_x_list)

    def __getitem__(self, idx):
        x_seq = torch.tensor(self.seq_x_list[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x_seq, y

# =========================================================
# NORMALIZATION
# =========================================================
def compute_train_stats(dataset_subset):
    xs = []

    for x_seq, _ in dataset_subset:
        xs.append(x_seq.numpy())

    X = np.concatenate(xs, axis=0)  # (all_windows, C, T)

    x_mean = X.mean(axis=(0, 2), keepdims=True)
    x_std = X.std(axis=(0, 2), keepdims=True) + 1e-6

    return x_mean.astype(np.float32), x_std.astype(np.float32)

class NormalizedClassificationDataset(Dataset):
    def __init__(self, base_dataset, x_mean, x_std):
        self.base_dataset = base_dataset
        self.x_mean = torch.tensor(x_mean, dtype=torch.float32)
        self.x_std = torch.tensor(x_std, dtype=torch.float32)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x_seq, y = self.base_dataset[idx]

        x_seq = (x_seq - self.x_mean) / self.x_std
        x_seq = x_seq - x_seq.mean(dim=-1, keepdim=True)
        x_seq = x_seq / (x_seq.std(dim=-1, keepdim=True) + 1e-6)
        x_seq = x_seq / (x_seq.abs().max(dim=-1, keepdim=True)[0] + 1e-6)

        return x_seq, y

# =========================================================
# MODEL: CNN + GRU
# =========================================================
class PlainCNNEncoder(nn.Module):
    def __init__(self, in_channels=143, feature_dim=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(feature_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.net(x)
        x = x.squeeze(-1)
        x = self.dropout(x)
        return x

class CNNGRUClassifier(nn.Module):
    def __init__(self, in_channels=143, cnn_feature_dim=256, gru_hidden=256, dropout=0.2):
        super().__init__()

        self.encoder = PlainCNNEncoder(
            in_channels=in_channels,
            feature_dim=cnn_feature_dim,
            dropout=dropout
        )

        self.temporal_gru = nn.GRU(
            input_size=cnn_feature_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.head = nn.Sequential(
            nn.Linear(gru_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_seq):
        # x_seq: [B, W, C, T]
        B, W, C, T = x_seq.shape
        x = x_seq.reshape(B * W, C, T)

        z = self.encoder(x)           # [B*W, F]
        z = z.reshape(B, W, -1)       # [B, W, F]

        z_gru, _ = self.temporal_gru(z)   # [B, W, 2H]
        z_pool = z_gru.mean(dim=1)        # [B, 2H]

        logits = self.head(z_pool).squeeze(1)   # [B]
        return logits

# =========================================================
# METRICS
# =========================================================
def compute_classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)
    return acc, f1

def find_best_threshold(y_true, y_prob):
    best_thr = 0.5
    best_f1 = -1.0

    for thr in np.arange(0.30, 0.71, 0.02):
        y_pred = (y_prob >= thr).astype(np.int64)
        f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)

    return best_thr, best_f1

# =========================================================
# TRAIN / EVAL: CNN + GRU
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x_seq)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return {"loss": total_loss / len(loader)}

@torch.no_grad()
def evaluate_probs(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_true = []
    all_prob = []

    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x_seq)
        loss = criterion(logits, y)
        total_loss += loss.item()

        probs = torch.sigmoid(logits)

        all_true.append(y.cpu().numpy())
        all_prob.append(probs.cpu().numpy())

    y_true = np.concatenate(all_true).astype(np.int64)
    y_prob = np.concatenate(all_prob)

    return {
        "loss": total_loss / len(loader),
        "y_true": y_true,
        "y_prob": y_prob,
    }

@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.5):
    raw = evaluate_probs(model, loader, criterion)

    y_true = raw["y_true"]
    y_prob = raw["y_prob"]
    y_pred = (y_prob >= threshold).astype(np.int64)

    acc, f1 = compute_classification_metrics(y_true, y_pred)

    return {
        "loss": raw["loss"],
        "accuracy": acc,
        "f1": f1,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

# =========================================================
# CURVE PLOTTING
# =========================================================
def save_fold_curves(history, held_out_subject):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Arousal Loss Curves - Held-out {held_out_subject}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"curve_loss_loso_arousal_gru_{held_out_subject}.png")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_acc"], label="Val Accuracy")
    plt.plot(epochs, history["val_f1"], label="Val F1")
    plt.xlabel("Epoch")
    plt.ylabel("Metric")
    plt.title(f"Arousal Metrics - Held-out {held_out_subject}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"curve_metrics_loso_arousal_gru_{held_out_subject}.png")
    plt.close()

# =========================================================
# FEATURES FOR RF / XGBOOST
# =========================================================
def extract_trial_features(seq_x_list):
    features = []

    for x_seq in seq_x_list:
        # x_seq: [W, C, T]
        mean_feat = x_seq.mean(axis=(0, 2))       # [C]
        std_feat = x_seq.std(axis=(0, 2))         # [C]
        max_feat = np.abs(x_seq).max(axis=(0, 2)) # [C]

        feat = np.concatenate([mean_feat, std_feat, max_feat], axis=0)
        features.append(feat.astype(np.float32))

    return np.stack(features, axis=0)

# =========================================================
# TRAIN ONE LOSO FOLD: CNN+GRU
# =========================================================
def train_one_loso_fold_gru(held_out_subject, seq_x_list, y_list, subject_list):
    train_subjects = sorted(list(set(subject_list) - {held_out_subject}))
    val_subjects = [held_out_subject]

    train_x, train_y_raw = filter_by_subjects(seq_x_list, y_list, subject_list, set(train_subjects))
    val_x, val_y_raw = filter_by_subjects(seq_x_list, y_list, subject_list, set(val_subjects))

    arousal_thresh = compute_train_arousal_threshold(train_y_raw)

    train_labels = make_arousal_binary_labels(train_y_raw, arousal_thresh)
    val_labels = make_arousal_binary_labels(val_y_raw, arousal_thresh)

    print("\n" + "=" * 70)
    print(f"LOSO FOLD | Held-out subject: {held_out_subject} | Task: arousal")
    print("=" * 70)
    print(f"Train trials: {len(train_x)}")
    print(f"Val trials:   {len(val_x)}")
    print(f"Train threshold arousal: {arousal_thresh:.4f}")
    print(f"Train class counts: {np.bincount(train_labels, minlength=2)}")
    print(f"Val class counts:   {np.bincount(val_labels, minlength=2)}")

    train_base = TrialClassificationDataset(train_x, train_labels)
    val_base = TrialClassificationDataset(val_x, val_labels)

    x_mean, x_std = compute_train_stats(train_base)
    train_set = NormalizedClassificationDataset(train_base, x_mean, x_std)
    val_set = NormalizedClassificationDataset(val_base, x_mean, x_std)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = CNNGRUClassifier(
        in_channels=143,
        cnn_feature_dim=256,
        gru_hidden=256,
        dropout=0.2
    ).to(DEVICE)

    class_counts = np.bincount(train_labels, minlength=2).astype(np.float32)
    neg_count, pos_count = class_counts[0], class_counts[1]
    pos_weight_value = neg_count / max(pos_count, 1.0)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    best_score = -float("inf")
    best_epoch = -1
    best_checkpoint = None
    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
        "val_thr": [],
    }

    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion)

        val_raw = evaluate_probs(model, val_loader, criterion)
        epoch_thr, _ = find_best_threshold(val_raw["y_true"], val_raw["y_prob"])
        val_metrics = evaluate(model, val_loader, criterion, threshold=epoch_thr)

        score = val_metrics["f1"]
        scheduler.step(score)

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_f1"].append(val_metrics["f1"])
        history["val_thr"].append(epoch_thr)

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Held-out {held_out_subject} | arousal | Epoch {epoch:03d} | "
            f"LR: {current_lr:.6f} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Acc: {val_metrics['accuracy']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f} | "
            f"Thr: {epoch_thr:.2f}"
        )

        improved = score > (best_score + MIN_DELTA)

        if improved:
            best_score = score
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "held_out_subject": held_out_subject,
                "epoch": epoch,
                "score": score,
                "accuracy": val_metrics["accuracy"],
                "f1": val_metrics["f1"],
                "threshold": epoch_thr,
                "y_true": val_metrics["y_true"],
                "y_pred": val_metrics["y_pred"],
                "y_prob": val_metrics["y_prob"],
                "arousal_thresh": arousal_thresh,
            }

            torch.save(
                {
                    "held_out_subject": held_out_subject,
                    "task_name": "arousal",
                    "epoch": epoch,
                    "model_state_dict": copy.deepcopy(model.state_dict()),
                    "accuracy": val_metrics["accuracy"],
                    "f1": val_metrics["f1"],
                    "threshold": epoch_thr,
                    "arousal_thresh": arousal_thresh,
                },
                f"best_model_loso_arousal_gru_{held_out_subject}.pt"
            )

            print(
                f"🔥 Best GRU fold updated | "
                f"Epoch {epoch} | "
                f"Acc: {val_metrics['accuracy']:.4f} | "
                f"F1: {val_metrics['f1']:.4f} | "
                f"Thr: {epoch_thr:.2f}"
            )
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping for held-out {held_out_subject} at epoch {epoch}. "
                f"Best epoch was {best_epoch}."
            )
            break

    if best_checkpoint is None:
        raise RuntimeError(f"No best checkpoint saved for held-out subject {held_out_subject}")

    np.save(f"best_y_true_loso_arousal_gru_{held_out_subject}.npy", best_checkpoint["y_true"])
    np.save(f"best_y_pred_loso_arousal_gru_{held_out_subject}.npy", best_checkpoint["y_pred"])
    np.save(f"best_y_prob_loso_arousal_gru_{held_out_subject}.npy", best_checkpoint["y_prob"])

    save_fold_curves(history, held_out_subject)

    with open(f"history_loso_arousal_gru_{held_out_subject}.json", "w") as f:
        json.dump(to_python(history), f, indent=4)

    print("\nClassification report:")
    print(classification_report(best_checkpoint["y_true"], best_checkpoint["y_pred"], digits=4, zero_division=0))

    return {
        "held_out_subject": held_out_subject,
        "best_epoch": best_checkpoint["epoch"],
        "accuracy": best_checkpoint["accuracy"],
        "f1": best_checkpoint["f1"],
        "threshold": best_checkpoint["threshold"],
        "arousal_thresh": best_checkpoint["arousal_thresh"],
    }

# =========================================================
# TRAIN ONE LOSO FOLD: RF / XGBOOST
# =========================================================
def train_one_loso_fold_tree_models(held_out_subject, seq_x_list, y_list, subject_list):
    train_subjects = sorted(list(set(subject_list) - {held_out_subject}))
    val_subjects = [held_out_subject]

    train_x, train_y_raw = filter_by_subjects(seq_x_list, y_list, subject_list, set(train_subjects))
    val_x, val_y_raw = filter_by_subjects(seq_x_list, y_list, subject_list, set(val_subjects))

    arousal_thresh = compute_train_arousal_threshold(train_y_raw)
    train_labels = make_arousal_binary_labels(train_y_raw, arousal_thresh)
    val_labels = make_arousal_binary_labels(val_y_raw, arousal_thresh)

    # normalize using train stats
    train_base = TrialClassificationDataset(train_x, train_labels)
    x_mean, x_std = compute_train_stats(train_base)

    def normalize_seq_list(seq_list, x_mean, x_std):
        out = []
        for x_seq in seq_list:
            x = (x_seq - x_mean) / x_std
            x = x - x.mean(axis=-1, keepdims=True)
            x = x / (x.std(axis=-1, keepdims=True) + 1e-6)
            x = x / (np.abs(x).max(axis=-1, keepdims=True) + 1e-6)
            out.append(x.astype(np.float32))
        return out

    train_x_norm = normalize_seq_list(train_x, x_mean, x_std)
    val_x_norm = normalize_seq_list(val_x, x_mean, x_std)

    X_train = extract_trial_features(train_x_norm)
    X_val = extract_trial_features(val_x_norm)

    results = {}

    # -------------------------
    # Random Forest
    # -------------------------
    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=RUN_SEED,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf_model.fit(X_train, train_labels)
    rf_probs = rf_model.predict_proba(X_val)[:, 1]
    rf_thr, _ = find_best_threshold(val_labels, rf_probs)
    rf_pred = (rf_probs >= rf_thr).astype(np.int64)

    rf_acc, rf_f1 = compute_classification_metrics(val_labels, rf_pred)

    results["random_forest"] = {
        "accuracy": rf_acc,
        "f1": rf_f1,
        "threshold": rf_thr,
    }

    print(
        f"[RF] Held-out {held_out_subject} | "
        f"Acc: {rf_acc:.4f} | F1: {rf_f1:.4f} | Thr: {rf_thr:.2f}"
    )

    # -------------------------
    # XGBoost
    # -------------------------
    if HAS_XGBOOST:
        neg_count = np.sum(train_labels == 0)
        pos_count = np.sum(train_labels == 1)
        scale_pos_weight = neg_count / max(pos_count, 1)

        xgb_model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=RUN_SEED,
            n_jobs=-1
        )

        xgb_model.fit(X_train, train_labels)
        xgb_probs = xgb_model.predict_proba(X_val)[:, 1]
        xgb_thr, _ = find_best_threshold(val_labels, xgb_probs)
        xgb_pred = (xgb_probs >= xgb_thr).astype(np.int64)

        xgb_acc, xgb_f1 = compute_classification_metrics(val_labels, xgb_pred)

        results["xgboost"] = {
            "accuracy": xgb_acc,
            "f1": xgb_f1,
            "threshold": xgb_thr,
        }

        print(
            f"[XGB] Held-out {held_out_subject} | "
            f"Acc: {xgb_acc:.4f} | F1: {xgb_f1:.4f} | Thr: {xgb_thr:.2f}"
        )
    else:
        results["xgboost"] = None

    return results

# =========================================================
# RUN ALL LOSO
# =========================================================
def run_loso_arousal(seq_x_list, y_list, subject_list):
    unique_subjects = sorted(list(set(subject_list)))
    print(f"\nRunning LOSO arousal across {len(unique_subjects)} subjects:")
    print(unique_subjects)

    gru_fold_results = []
    rf_fold_results = []
    xgb_fold_results = []

    for held_out_subject in unique_subjects:
        # CNN + GRU
        gru_result = train_one_loso_fold_gru(
            held_out_subject,
            seq_x_list,
            y_list,
            subject_list
        )
        gru_fold_results.append(gru_result)

        # RF + XGB
        tree_results = train_one_loso_fold_tree_models(
            held_out_subject,
            seq_x_list,
            y_list,
            subject_list
        )

        rf_fold_results.append({
            "held_out_subject": held_out_subject,
            "accuracy": tree_results["random_forest"]["accuracy"],
            "f1": tree_results["random_forest"]["f1"],
            "threshold": tree_results["random_forest"]["threshold"],
        })

        if tree_results["xgboost"] is not None:
            xgb_fold_results.append({
                "held_out_subject": held_out_subject,
                "accuracy": tree_results["xgboost"]["accuracy"],
                "f1": tree_results["xgboost"]["f1"],
                "threshold": tree_results["xgboost"]["threshold"],
            })

    def summarize(fold_results, name):
        accs = np.array([r["accuracy"] for r in fold_results], dtype=np.float32)
        f1s = np.array([r["f1"] for r in fold_results], dtype=np.float32)

        return {
            "model": name,
            "accuracy_mean": float(accs.mean()),
            "accuracy_std": float(accs.std()),
            "f1_mean": float(f1s.mean()),
            "f1_std": float(f1s.std()),
            "per_subject": fold_results,
        }

    summary = {
        "cnn_gru_bce": summarize(gru_fold_results, "cnn_gru_bce"),
        "random_forest": summarize(rf_fold_results, "random_forest"),
        "xgboost": summarize(xgb_fold_results, "xgboost") if len(xgb_fold_results) > 0 else None,
    }

    with open("loso_arousal_baseline_comparison_summary.json", "w") as f:
        json.dump(to_python(summary), f, indent=4)

    print("\n" + "=" * 70)
    print("AROUSAL BASELINE COMPARISON SUMMARY")
    print("=" * 70)

    print("\nCNN + GRU + BCEWithLogitsLoss")
    print(f"Accuracy: {summary['cnn_gru_bce']['accuracy_mean']:.4f} ± {summary['cnn_gru_bce']['accuracy_std']:.4f}")
    print(f"F1:       {summary['cnn_gru_bce']['f1_mean']:.4f} ± {summary['cnn_gru_bce']['f1_std']:.4f}")

    print("\nRANDOM FOREST")
    print(f"Accuracy: {summary['random_forest']['accuracy_mean']:.4f} ± {summary['random_forest']['accuracy_std']:.4f}")
    print(f"F1:       {summary['random_forest']['f1_mean']:.4f} ± {summary['random_forest']['f1_std']:.4f}")

    if summary["xgboost"] is not None:
        print("\nXGBOOST")
        print(f"Accuracy: {summary['xgboost']['accuracy_mean']:.4f} ± {summary['xgboost']['accuracy_std']:.4f}")
        print(f"F1:       {summary['xgboost']['f1_mean']:.4f} ± {summary['xgboost']['f1_std']:.4f}")
    else:
        print("\nXGBOOST")
        print("Skipped because xgboost is not installed.")

    return summary

# =========================================================
# MAIN
# =========================================================
def main():
    print("Loading processed trials...")
    seq_x_list, y_list, subject_list = load_all_trials_with_subjects(DATA_DIR)

    if len(seq_x_list) == 0:
        raise FileNotFoundError(f"No processed trials found in {DATA_DIR}.")

    print("Example x_seq shape:", seq_x_list[0].shape)
    print("Example y shape:", y_list[0].shape)

    config = {
        "task": "arousal",
        "deep_model": "CNNGRUClassifier",
        "loss": "BCEWithLogitsLoss",
        "cnn_feature_dim": 256,
        "gru_hidden": 256,
        "dropout": 0.2,
        "lr": LR,
        "epochs": EPOCHS,
        "evaluation": "LOSO binary classification",
        "label_strategy": "global train-only threshold per fold",
        "threshold_type": "median",
        "classical_models": ["RandomForestClassifier", "XGBClassifier"],
    }

    with open("loso_arousal_baseline_comparison_config.json", "w") as f:
        json.dump(to_python(config), f, indent=4)

    summary = run_loso_arousal(seq_x_list, y_list, subject_list)

    with open("loso_arousal_baseline_comparison_final_summary.json", "w") as f:
        json.dump(to_python(summary), f, indent=4)

    print("\nSaved:")
    print("  loso_arousal_baseline_comparison_config.json")
    print("  loso_arousal_baseline_comparison_summary.json")
    print("  loso_arousal_baseline_comparison_final_summary.json")
    print("  best_model_loso_arousal_gru_<subject>.pt")
    print("  best_y_true_loso_arousal_gru_<subject>.npy")
    print("  best_y_pred_loso_arousal_gru_<subject>.npy")
    print("  best_y_prob_loso_arousal_gru_<subject>.npy")
    print("  history_loso_arousal_gru_<subject>.json")
    print("  curve_loss_loso_arousal_gru_<subject>.png")
    print("  curve_metrics_loso_arousal_gru_<subject>.png")

if __name__ == "__main__":
    main()

Loading processed trials...
Searching in: /content/drive/MyDrive/processed_meg_regression
Found 82 X files
Total usable trial sequences: 814
Total unique subjects: 21
Example x_seq shape: (4, 143, 2048)
Example y shape: (2,)

Running LOSO arousal across 21 subjects:
['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-20', 'sub-21', 'sub-22', 'sub-23']

LOSO FOLD | Held-out subject: sub-01 | Task: arousal
Train trials: 774
Val trials:   40
Train threshold arousal: 0.5000
Train class counts: [313 461]
Val class counts:   [10 30]
Held-out sub-01 | arousal | Epoch 001 | LR: 0.000100 | Train Loss: 0.5318 | Val Loss: 0.4971 | Val Acc: 0.7250 | Val F1: 0.8406 | Thr: 0.30
🔥 Best GRU fold updated | Epoch 1 | Acc: 0.7250 | F1: 0.8406 | Thr: 0.30
Held-out sub-01 | arousal | Epoch 002 | LR: 0.000100 | Train Loss: 0.4385 | Val Loss: 0.6820 | Val Acc: 0.6250 | Val F1: 0.7619 | 

In [ ]:
import shutil

# Define the destination directory in Google Drive
output_dir = os.path.join(DATA_DIR, 'subject_specific_binary_results')

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"Saving results to: {output_dir}")

Saving results to: /content/drive/MyDrive/processed_meg_regression/subject_specific_binary_results


In [ ]:
# List of files to move
files_to_move = [
    'subject_specific_binary_config.json',
    'subject_specific_binary_summary.json'
]

# Get all subject-specific files created during the run
all_generated_files = glob.glob('*_sub-??_*.pt') + \
                      glob.glob('*_sub-??_*.json') + \
                      glob.glob('*_sub-??_*.npy') + \
                      glob.glob('*_sub-??_*.png')

files_to_move.extend(all_generated_files)

# Move each file to the output directory
for f in files_to_move:
    if os.path.exists(f):
        shutil.move(f, os.path.join(output_dir, f))
    else:
        print(f"[WARN] File not found: {f}")

print("Files moved successfully.")

Files moved successfully.


In [ ]:
# List the files in the new directory to confirm
print("\nContents of the output directory:")
for item in os.listdir(output_dir):
    print(item)


Contents of the output directory:
subject_specific_binary_config.json
subject_specific_binary_summary.json
best_model_sub-01_valence_binary.pt
best_model_sub-19_arousal_binary.pt
best_model_sub-09_arousal_binary.pt
best_model_sub-18_valence_binary.pt
best_model_sub-02_valence_binary.pt
best_model_sub-08_arousal_binary.pt
best_model_sub-03_valence_binary.pt
best_model_sub-04_valence_binary.pt
best_model_sub-06_arousal_binary.pt
best_model_sub-02_arousal_binary.pt
best_model_sub-14_valence_binary.pt
best_model_sub-11_arousal_binary.pt
best_model_sub-10_arousal_binary.pt
best_model_sub-16_arousal_binary.pt
best_model_sub-15_arousal_binary.pt
best_model_sub-07_arousal_binary.pt
best_model_sub-15_valence_binary.pt
best_model_sub-20_valence_binary.pt
best_model_sub-06_valence_binary.pt
best_model_sub-08_valence_binary.pt
best_model_sub-09_valence_binary.pt
best_model_sub-23_valence_binary.pt
best_model_sub-10_valence_binary.pt
best_model_sub-21_valence_binary.pt
best_model_sub-16_valence_bi